In [2]:
import sys
!git clone -b text-detection-model https://github.com/duckysmacky/cogniflex.git
%cd cogniflex
%cd /kaggle/working/cogniflex
!pwd
!ls model
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".."))) 
sys.path.insert(0, './model')

Cloning into 'cogniflex'...
remote: Enumerating objects: 1256, done.
remote: Counting objects: 100% (570/570), done.
remote: Compressing objects: 100% (249/249), done.
remote: Total 1256 (delta 245), reused 476 (delta 200), pack-reused 686 (from 1)
Receiving objects: 100% (1256/1256), 80.39 MiB | 42.85 MiB/s, done.
Resolving deltas: 100% (473/473), done.
/kaggle/working/cogniflex
/kaggle/working/cogniflex
/kaggle/working/cogniflex
notebooks  requirement_docker.txt  requirements.txt  utils


In [3]:
import torch.nn as nn
import os
from PIL import Image
from torchvision import transforms
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import sys
from sklearn.metrics import f1_score

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".."))) 
from utils.text_utils.text_preprocess_torch import tokenize_texts, AIDetectionDataset, RoBERTaDetector

In [4]:
BATCH_SIZE = 16
MODEL_NAME = 'roberta-base'

In [5]:
os.listdir('/kaggle/input/datasets/shanegerami/ai-vs-human-text')

['AI_Human.csv']

In [6]:
# dataset = pd.read_csv('/kaggle/input/datasets/...')
dataset = pd.read_csv('/kaggle/input/datasets/shanegerami/ai-vs-human-text/AI_Human.csv')
dataset = dataset[::5].reset_index(drop=True)

import gc
gc.collect()

data, labels = dataset['text'], dataset['generated'].astype(int) #1 = ai, 0 = human
data_train, data_test, labels_train, labels_test = train_test_split(data, labels, test_size=0.2, random_state=42)
print(len(labels), len(data))
print(labels[labels == 1].sum(), labels[labels == 0].count())

97447 97447
36016 61431


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = AIDetectionDataset(
    data_train, labels_train, tokenizer
)

test_dataset = AIDetectionDataset(
    data_test, labels_test, tokenizer
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,         
    num_workers=2         
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False       
)

In [9]:
"""
Далее происходит подготовка текстов и fine tuning модели RoBERTa
Подготовка текста - просто разделение его на токены и группировка по батчам 
с помощью AIDetectionDataset

Векторизация происходит автоматически в первых слоях RoBERTa
"""


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
roberta = RoBERTaDetector(MODEL_NAME, 0.2) #custom model from text_utils


optimizer = torch.optim.AdamW(roberta.parameters(), lr=2e-5)
weights = torch.tensor([1.0, 61431/36016], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)

roberta = roberta.to(device)
print(f'device: {device}')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device: cuda


In [10]:
def train_model_with_early_stopping(num_epochs=5):

    best_val_loss = float('inf')
    patience = 2
    epochs_no_improve = 0
    save_path = 'best_roberta_fc.pth'

    for epoch in range(num_epochs):

        print(f"Epoch {epoch+1}")
        roberta.train()
        train_loss = 0
        train_batches = 0

        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = roberta(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(roberta.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            train_batches += 1
            #print(train_batches)

        avg_train_loss = train_loss/train_batches
        print(f'Train loss: {avg_train_loss:.4f}')

        roberta.eval()
        val_loss = 0

        pred_epoch = []
        labels_epoch = []
        
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = roberta(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                pred = torch.argmax(outputs, dim=1)
                pred_epoch.extend(pred.detach().cpu().numpy())
                labels_epoch.extend(labels.cpu().numpy())
            

        avg_val_loss = val_loss/len(test_loader.dataset)
        f1score = f1_score(labels_epoch, pred_epoch)

        print(f"Val_loss: {avg_val_loss:.4f}")
        print(f"f1score: {f1score:.4f}")

        #early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0

            torch.save(roberta.state_dict(), save_path)
            print(f'Model saved at epoch {epoch+1}')
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print('There is no improvement')
            break

        scheduler.step()

In [11]:
train_model_with_early_stopping(2)

Epoch 1
Train loss: 0.0369
Val_loss: 0.0013
f1score: 0.9944
Model saved at epoch 1
Epoch 2


KeyboardInterrupt: 

In [13]:
from IPython.display import FileLink

FileLink('/kaggle/working/cogniflex/best_roberta_fc.pth')

/kaggle/working/cogniflex/best_roberta_fc.pth